<a href="https://colab.research.google.com/github/ajndantas/Conselheiro-de-Saude-Mental/blob/master/Conselheiro_de_Saude_Mental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install google-genai
%pip -q install google-adk

In [ ]:
from os import environ
from google.colab import userdata

api_key = 'GOOGLE_API_KEY'
environ[api_key] = userdata.get(api_key)

In [ ]:
from google import genai

client = genai.Client()

MODEL_ID = "gemini-2.0-flash"

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types
from datetime import date
import textwrap
from IPython.display import display, Markdown
import requests
import warnings

warnings.filterwarnings("ignore")

In [ ]:
def call_agent(agent: Agent, message_text: str) -> str:
    session_service = InMemorySessionService()
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""

    for event in runner.run(user_id="user1", session_id="session1", new_message=content):

        if event.is_final_response():
            for part in event.content.parts:
                if part.text is not None:

                    final_response += part.text

                    final_response += "\n"

    return final_response

In [ ]:
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
##########################################
# ---     Agente 1: Psicólogo        --- #
##########################################

def agente_psicologo(resposta):
    psicologo = Agent(
        # Inserir as instruções do Agente Planejador #################################################
        name="psicologo",
        model="gemini-2.0-flash",
        instruction="""
        Aja como um psicólogo e seja empático e suscinto. Você deve fazer o seguinte:
        1 - Dar conselhos para alguém que está com a saúde mental debilitada conforme a resposta dada.
        2 - Se a resposta informada for um sentimento positivo, responda para que a pessoa aproveite a vida e seja feliz.
        3 - Informar 5 possíveis transtornos mentais causadores da resposta dada usando o Google (google_search). Se a transtorno mental tiver poucas reações entusiasmadas, é possível que ele não seja tão relevante assim e
        possa ser substituída por outra que tenha mais.
        """,
        description="Agente psicólogo",
        tools=[google_search] # Utilizando a ferramenta de busca do Google importada anteriormente do ADK
    )

    entrada_do_agente_psicologo = f"Resposta:{resposta}\n"
    conselhos = call_agent(psicologo, entrada_do_agente_psicologo)

    return conselhos

In [ ]:
################################################
# --- Agente 2: Atividades --- #
################################################

def agente_atividades(conselhos):
    atividades = Agent(
        name="atividades",
        model="gemini-2.0-flash",
        instruction="""
        Você é um psicologo, e reaja de maneira empática e suscinta. Você deve:
        1 - Ser capaz de sugerir no máximo 5 atividades de relaxamento ou mindfullness.
        Se a atividade tiver poucas reações entusiasmadas, é possível que ele não seja tão relevante assim e
        possa ser substituída por outra que tenha mais.
        """,
        description="Lista atividades de relaxamento e mindfullness",
        tools=[google_search]
    )

    entrada_do_agente_atividades = f"Conselhos:{conselhos}\n"
    atividades = call_agent(atividades, entrada_do_agente_atividades)

    return atividades



In [ ]:
print("🚀 Iniciando o Conselheiro de Saúde Mental 🚀")

resposta = input("❓ Como você está se sentindo hoje ? ")

if not resposta:
    print("Nenhuma resposta foi fornecida")
else:
    print(f"🔍 Vamos então obter os conselhos de nosso psicólogo virtual em função do que você respondeu {resposta}")

    conselhos = agente_psicologo(resposta)
    print("\n 📝 Conselhos\n")
    display(to_markdown(conselhos))
    print("-------------------------------------------------")

    atividades = agente_atividades(conselhos)
    print("\n 📝 Atividades\n")
    display(to_markdown(atividades))
    print("-------------------------------------------------")